In [1]:
# Install all wheels from the input notebook's package directory
!pip install --no-index --find-links=/kaggle/input/notebooks/harshankmatkar/dependencies/packages tinker-cookbook


Looking in links: /kaggle/input/notebooks/harshankmatkar/dependencies/packages
Processing /kaggle/input/notebooks/harshankmatkar/dependencies/packages/tinker_cookbook-0.3.1.dev13+g8fc8c1758-py3-none-any.whl
Processing /kaggle/input/notebooks/harshankmatkar/dependencies/packages/chz-0.4.0-py3-none-any.whl (from tinker-cookbook)
Processing /kaggle/input/notebooks/harshankmatkar/dependencies/packages/tinker-0.18.0-py3-none-any.whl (from tinker-cookbook)


In [2]:
import torch
import tinker_cookbook.weights._adapter as A

FORCED_FUSED_RANK = 32

def _compress_lora_pair_to_rank(B: torch.Tensor, A_mat: torch.Tensor, rank: int):
    # Delta = B @ A, shape [out_dim, in_dim]
    delta = B.float() @ A_mat.float()

    # Best rank-k approximation in Frobenius norm
    U, S, Vh = torch.linalg.svd(delta, full_matrices=False)
    U = U[:, :rank]
    S = S[:rank]
    Vh = Vh[:rank, :]

    sroot = torch.sqrt(S)
    B_new = U * sroot.unsqueeze(0)          # [out_dim, rank]
    A_new = sroot.unsqueeze(1) * Vh         # [rank, in_dim]

    return B_new.to(B.dtype).contiguous(), A_new.to(A_mat.dtype).contiguous()


def patched_merge_fused_projections(
    fused_model_key: str,
    adapter_layer_prefix: str,
    components,
    model_state_shapes,
    peft_weights,
    target_modules,
    profile,
) -> int:
    fused_out_dim = model_state_shapes[fused_model_key][0]
    fused_target_name = fused_model_key.removesuffix(".weight").rsplit(".", 1)[-1]

    component_order = None
    for target, comps in profile.fused_projection_map:
        if target == fused_target_name:
            component_order = comps
            break
    assert component_order is not None

    comp_by_name = {name: (lora_A, lora_B) for name, lora_A, lora_B in components}

    lora_A_parts = []
    comp_slices = []   # (row_start, row_end, rank)
    merged_rank = 0
    row_offset = 0

    for comp_name in component_order:
        if comp_name not in comp_by_name:
            raise RuntimeError(
                f"Missing component {comp_name!r} for fused target {fused_model_key!r}"
            )
        lora_A, lora_B = comp_by_name[comp_name]
        r = lora_A.shape[0]
        out_dim = lora_B.shape[0]

        lora_A_parts.append(lora_A)
        comp_slices.append((row_offset, row_offset + out_dim, r))
        row_offset += out_dim
        merged_rank += r

    merged_lora_A = torch.cat(lora_A_parts, dim=0)
    merged_lora_B = torch.zeros(
        fused_out_dim, merged_rank, dtype=merged_lora_A.dtype, device=merged_lora_A.device
    )

    rank_offset = 0
    for i, (row_start, row_end, r) in enumerate(comp_slices):
        _, lora_B = comp_by_name[component_order[i]]
        merged_lora_B[row_start:row_end, rank_offset:rank_offset + r] = lora_B
        rank_offset += r

    # Force fused modules back down to rank 32
    final_rank = merged_rank
    if merged_rank > FORCED_FUSED_RANK:
        merged_lora_B, merged_lora_A = _compress_lora_pair_to_rank(
            merged_lora_B, merged_lora_A, FORCED_FUSED_RANK
        )
        final_rank = FORCED_FUSED_RANK

    peft_target_key = f"{adapter_layer_prefix}.{fused_target_name}.weight"
    A._add_peft_weight(peft_target_key, merged_lora_A, merged_lora_B, peft_weights, target_modules)
    return final_rank


# monkey-patch
A._merge_fused_projections = patched_merge_fused_projections
print("patched:", A._merge_fused_projections.__name__)

patched: patched_merge_fused_projections


In [3]:
from tinker_cookbook import weights

weights.build_lora_adapter(
    base_model="/kaggle/input/models/metric/nemotron-3-nano-30b-a3b-bf16/transformers/default/1",
    adapter_path="/kaggle/input/models/huikang/nemotron-adapter/transformers/default/20",
    output_path="/kaggle/working/nemotron-adapter-ready-to-submit",
)

MoE expert LoRA serving for nemotron models is experimental in vLLM and not yet supported in SGLang. The adapter will be produced but may not work with all serving configurations.


In [4]:
import shutil
shutil.make_archive('/kaggle/working/submission', 'zip', '/kaggle/working/nemotron-adapter-ready-to-submit')

'/kaggle/working/submission.zip'